<a href="https://colab.research.google.com/github/profsuccodifrutta/ai_act_RAG_navigator/blob/main/public_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CREATION OF ENVIORMENT

In [1]:
# --- CELL 1: GITHUB REPOSITORY SETUP ---
import os

# Configuration for GitHub
GIT_USER = "profsuccodifrutta"
GIT_REPO = "ai_act_RAG_navigator"
REPO_URL = f"https://github.com/{GIT_USER}/{GIT_REPO}.git"

# Clone repo
if not os.path.exists(GIT_REPO):
    print(f"Cloning repository {GIT_REPO}...")
    !git clone {REPO_URL}
else:
    print(f"Folder {GIT_REPO} already exists.")

# Move into the repo directory
%cd {GIT_REPO}

Cloning repository ai_act_RAG_navigator...
Cloning into 'ai_act_RAG_navigator'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 154 (delta 90), reused 12 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 2.59 MiB | 18.54 MiB/s, done.
Resolving deltas: 100% (90/90), done.
/content/ai_act_RAG_navigator


In [2]:
#  SYSTEM SETUP and DEPENDENCIES
!curl -LsSf https://astral.sh/uv/install.sh | sh
import sys
import os

# Add uv to PATH
sys.path.append("/root/.cargo/bin")

# Initialize uv
if not os.path.exists("pyproject.toml"):
    !uv init
else:
    print("uv already initialized.")


!uv add langchain langchain-community langchain-google-genai faiss-cpu pymupdf spacy python-dotenv langchain-text-splitters langchain-core datasets sentence-transformers rank_bm25 pandas matplotlib scikit-learn plotly scipy tensorflow accelerate
!uv pip install --system langchain langchain-community langchain-google-genai faiss-cpu pymupdf spacy python-dotenv langchain-text-splitters langchain-core datasets sentence-transformers rank_bm25 pandas matplotlib scikit-learn plotly scipy tensorflow accelerate
print("\nEnvironment configuration completed!")

downloading uv 0.10.12 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!
uv already initialized.
Installing libraries... This may take a few minutes due to TensorFlow and PyTorch.
Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 171 packages in 2.55s
Prepared 168 packages in 3m 28s
Installed 168 packages in 4.77s
 + absl-py==2.4.0
 + accelerate==1.13.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.12.1
 + astunparse==1.6.3
 + attrs==25.4.0
 + blis==1.3.3
 + catalogue==2.0.10
 + certifi==2026.1.4
 + cffi==2.0.0
 + charset-normalizer==3.4.4
 + click==8.3.1
 + cloudpathlib==0.23.0
 + confection==0.1.5
 + contourpy==1.3.3
 + cryptography==46.0.5
 + cuda-bindings==12.9.4
 + cuda-pathfinder==1.4.3
 + cycler==0.12.1
 + cymem==2.0.13
 + dataclasses-json==0.6.7
 + datasets==4.8.3
 + dill==0.4.1
 + di

# LOADING LIBRARIES

In [3]:
# --- CELL 3: IMPORTS ---
from google.colab import drive
import os
import requests
import shutil
import fitz  # This is PyMuPDF
import re
import pandas as pd
import matplotlib.pyplot as plt
import gc
import random
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
import spacy
from spacy import displacy
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import plotly.graph_objects as go
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import plotly.io as pio
import plotly.express as px
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from scipy.special import softmax
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from collections import defaultdict
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, TextVectorization, Input, Bidirectional, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import PolynomialDecay
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

print("All libraries successfully imported!")

All libraries successfully imported!


In [4]:
# --- CELL 3: DATA & RAG CONFIGURATION ---
import os

# Target the PDF directly in the repository root
local_pdf_path = "ai_act_2024.pdf"

if os.path.exists(local_pdf_path):
    print(f"PDF successfully found locally at: {local_pdf_path}")
else:
    print(f"Error: The file '{local_pdf_path}' was not found.")
    print("Please verify the exact name on GitHub and ensure you ran Cell 1.")

# Local persistence folder for FAISS index
local_persistence_path = "ai_act_rag_index"
if not os.path.exists(local_persistence_path):
    os.makedirs(local_persistence_path)
    print("Folder for index persistence ready locally.")
else:
    print("Folder for index persistence already exists.")

PDF successfully found locally at: ai_act_2024.pdf
Folder for index persistence ready locally.


# PRE-PROCESSING and EDA

The parse_ai_act_robust() function acts as a highly specialized document parser designed to extract the EU AI Act into clean, semantically isolated legal units. First, it reads the PDF page by page, stripping away formatting noise like document links, page numbers, and line-break hyphens. It employs a multiline regex strategy to detect and truncate footnote blocks at the bottom of the pages; this prevents those footnotes from being falsely extracted as standalone Recitals or corrupting the flow of the main text. After sanitizing the pages, it groups them into "RECITAL", "ARTICLE", and "ANNEX" buckets based on their structural locations in the document. Finally, the function puts the cleaned text back together and splits it only where real section titles appear (like “Article 5” on its own line). It does not split when those words appear inside a sentence (for example, “in accordance with Article 99”). This ensures that each row in the final dataframe contains one complete legal section, making it ideal for use in a RAG system.

In [32]:
def parse_ai_act_robust(pdf_path):
    doc = fitz.open(pdf_path)

    categories_text = {
        "RECITAL": [],
        "ARTICLE": [],
        "ANNEX": []
    }

    noise_pattern = re.compile(r"(\d+/144|ELI: http://data\.europa\.eu/eli/reg/.+|OJ L, 12\.7\.2024|EN|\(Text with EEA relevance\))")
    footnote_cleaner = re.compile(r"\n\s*\(\s*\d+\s*\)\s*(?:OJ\b|Position\b|Directive\b|Regulation\b|See\b|Case\b).*$", re.IGNORECASE | re.DOTALL)

    footnotes_removed = 0

    for page_num in range(len(doc)):
        text = doc[page_num].get_text("text").replace("-\n", "")

        clean_lines = []
        for line in text.split("\n"):
            cleaned_line = noise_pattern.sub("", line).strip()
            if cleaned_line and not cleaned_line.isdigit():
                clean_lines.append(cleaned_line)

        # Join the page back together
        final_page_text = "\n" + "\n".join(clean_lines)

        if footnote_cleaner.search(final_page_text):
            # Replace the footnote block with a single newline
            final_page_text = footnote_cleaner.sub("\n", final_page_text)
            footnotes_removed += 1

        p_idx = page_num + 1
        if p_idx < 44:
            categories_text["RECITAL"].append(final_page_text)
        elif 44 <= p_idx <= 123:
            categories_text["ARTICLE"].append(final_page_text)
        else:
            categories_text["ANNEX"].append(final_page_text)

    doc.close()
    print(f"Removed footnote blocks from {footnotes_removed} pages.")

    units = []

    # proces recitals
    recital_text = "\n" + "\n".join(categories_text["RECITAL"]) + "\n"
    rec_parts = re.split(r"\n\s*\(\s*(\d+)\s*\)\s+", recital_text)
    for i in range(1, len(rec_parts), 2):
        text_content = rec_parts[i+1].replace("\n", " ").strip()
        if text_content:
            units.append({"id": f"RECITAL {rec_parts[i]}", "category": "RECITAL", "text": text_content})

    # proces articles
    article_text = "\n" + "\n".join(categories_text["ARTICLE"]) + "\n"
    art_parts = re.split(r"(?i)\n\s*Article\s+(\d+)\s*\n", article_text)
    for i in range(1, len(art_parts), 2):
        text_content = art_parts[i+1].replace("\n", " ").strip()
        if text_content:
            units.append({"id": f"ARTICLE {art_parts[i]}", "category": "ARTICLE", "text": text_content})

    # proces annexes
    annex_text = "\n" + "\n".join(categories_text["ANNEX"]) + "\n"
    annex_parts = re.split(r"(?i)\n\s*ANNEX\s+([IVXLCDM]+)\s*\n", annex_text)
    for i in range(1, len(annex_parts), 2):
        text_content = annex_parts[i+1].replace("\n", " ").strip()
        if text_content:
            units.append({"id": f"ANNEX {annex_parts[i].upper()}", "category": "ANNEX", "text": text_content})

    return pd.DataFrame(units)

In [33]:
df_units = parse_ai_act_robust(local_pdf_path)
df_units['len'] = df_units['text'].str.len()

print(f"Clean Units created: {len(df_units)}")
print(f"Max length: {df_units['len'].max()}")

Removed footnote blocks from 23 pages.
Clean Units created: 307
Max length: 17095


An embedding vector represents the "average" meaning of a text. If a chunk is too long and covers five different legal requirements, the vector becomes vague, making it harder for the system to find that specific information during a search. Even if an LLM can read a huge chunk, providing too much irrelevant text alongside the answer creates "noise." This increases the risk of the model hallucinating or missing the specific detail you're looking for. Later on the lenght of the chunks will be calibrated, below is shown a visual inspection of the actual distribution.

In [34]:
avg_len = df_units['len'].mean()

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=df_units['len'],
    nbinsx=50,
    marker_color='#e74c3c',
    marker_line=dict(color='white', width=1),
    name='Unit Lengths'
))

fig.add_vline(x=1500, line_dash="dash", line_color="#c0392b")

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    line=dict(color='#c0392b', dash='dash'),
    name='Target RAG Limit (1500)'
))

fig.add_vline(x=avg_len, line_dash="solid", line_color="#f39c12")

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    line=dict(color='#f39c12', shape='linear'),
    name=f'Average Length ({avg_len:.0f})'
))

fig.update_layout(
    title='<b>Distribution of Legal Unit Lengths </b>',
    xaxis_title="Characters",
    yaxis_title="Number of Units",
    template='plotly_white',
    bargap=0.05,
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99,
        bgcolor="rgba(255, 255, 255, 0.5)"
    ),
    height=500
)


fig.show()

# METADATA ENRICHMENT WITH ERROR ANALYSIS

**NER FOR ENRICHING THE METADATA**


In a standard RAG system, the database relies entirely on dense vector embeddings to find relevant information. While vectors are excellent at understanding the semantic meaning or general theme of a question, they struggle significantly with exact keyword matching. For example, a vector search might easily confuse the legal obligations of a "Provider" with those of a "Deployer" because both concepts occupy a very similar mathematical space in the vector database.

**ERROR ANALYSIS METHODOLOGY**


Identifying and extracting these entities requires an iterative methodology that we'll call Error Analysis, as relying entirely on pre-trained machine learning models, this case of domain-specific legal text will lead to systemic failures. Pre-trained NLP models calculate statistical probabilities based on everyday internet text, they get easily confused by the dense capitalization, roman numerals, and acronyms found in the AI Act.

To overcome this, we established an iterative feedback loop:


1.   Baseline Definition: We injected a baseline set of deterministic rules (the EntityRuler) into the pipeline before the machine learning component.
2.   Visual Diagnostics: We ran random samples of the text through the pipeline and used a visualizer (displaCy) to inspect the model's behavior.
3. Identifying False Positives and Negatives: By reading the annotated text, we spotted areas where the model missed critical terms.
4. Rule Calibration: We updated our deterministic rules to either explicitly classify these edge cases (e.g. assigning "Annexes" to DOCUMENT)


The rules that drive this extraction are defined in a simple but highly powerful list called patterns. Every rule is a dictionary containing two key-value pairs: the "label" (which dictates the category, like "ORG" or "RISK_LEVEL") and the "pattern" (which defines what text to look for).

Afther the pattern have been identified they are added to the document object of langchain library, since could be useful to exploit these information while retriving, possibly increasing the rag system performance.

In [35]:
nlp = spacy.load("en_core_web_sm")

ruler = nlp.add_pipe("entity_ruler", before="ner") # ner = spacy module for ner
                                                   # before the model prediction

patterns = [
    # organizations and authoroties
    {"label": "ORG", "pattern": "AI Office"},
    {"label": "ORG", "pattern": "European Artificial Intelligence Board"},
    {"label": "ORG", "pattern": "Market Surveillance Authority"},
    {"label": "ORG", "pattern": [{"lower": "european"}, {"lower": "commission"}]},
    {"label": "ORG", "pattern": "European Data Protection Board"},
    {"label": "ORG", "pattern": "European Data Protection Supervisor"},
    {"label": "ORG", "pattern": "EDPB"},
    {"label": "ORG", "pattern": "EDPS"},
    {"label": "ORG", "pattern": "Board"},
    {"label": "ORG", "pattern": "Commission"},

    # geopolitical entities
    {"label": "GPE", "pattern": "Union"},
    {"label": "GPE", "pattern": "EU"},

    # documents
    {"label": "DOCUMENT", "pattern": "Regulation"},
    {"label": "DOCUMENT", "pattern": "Directive"},
    {"label": "DOCUMENT", "pattern": [{"lower": "annex"}, {"IS_ALPHA": True, "IS_UPPER": True}]},
    {"label": "DOCUMENT", "pattern": [{"lower": "annexes"}, {"IS_ALPHA": True, "IS_UPPER": True}]},

    # Chapters
    {"label": "LAW", "pattern": [{"lower": "chapter"}, {"IS_DIGIT": True}]},
    {"label": "LAW", "pattern": [{"lower": "chapter"}, {"IS_ALPHA": True, "IS_UPPER": True}]},

    # Sections
    {"label": "LAW", "pattern": [{"lower": "section"}, {"IS_DIGIT": True}]},
    {"label": "LAW", "pattern": [{"lower": "section"}, {"IS_ALPHA": True, "IS_UPPER": True}]},
    {"label": "LAW", "pattern": [{"lower": "sections"}, {"IS_DIGIT": True}]},
    {"label": "LAW", "pattern": [{"lower": "sections"}, {"IS_ALPHA": True, "IS_UPPER": True}]},

    # Articles
    {"label": "LAW", "pattern": [{"lower": "article"}, {"IS_DIGIT": True}]},

    # ignore these patterns
    {"label": "IGNORE", "pattern": "AI"},
    {"label": "IGNORE", "pattern": [{"lower": "artificial"}, {"lower": "intelligence"}]},
    {"label": "IGNORE", "pattern": [{"lower": "ai"}, {"lower": "system"}]},

    # concepts and legal roles
    {"label": "CONCEPT", "pattern": "CE"},
    {"label": "CONCEPT", "pattern": [{"text": "CE"}, {"lower": "marking"}]},
    {"label": "CONCEPT", "pattern": [{"lower": "general"}, {"IS_PUNCT": True, "OP": "?"}, {"lower": "purpose"}, {"lower": "ai"}]},
    {"label": "LEGAL_ROLE", "pattern": [{"lower": "provider"}]},
    {"label": "LEGAL_ROLE", "pattern": [{"lower": "providers"}]},
    {"label": "LEGAL_ROLE", "pattern": [{"lower": "deployer"}]},
    {"label": "LEGAL_ROLE", "pattern": [{"lower": "deployers"}]},
    # risk levels
    {"label": "RISK_LEVEL", "pattern": [{"lower": "high"}, {"IS_PUNCT": True, "OP": "?"}, {"lower": "risk"}]},
    {"label": "RISK_LEVEL", "pattern": [{"lower": "unacceptable"}, {"lower": "risk"}]},
    {"label": "RISK_LEVEL", "pattern": [{"lower": "limited"}, {"lower": "risk"}]},
    {"label": "RISK_LEVEL", "pattern": [{"lower": "minimal"}, {"lower": "risk"}]},
]

ruler.add_patterns(patterns)

**LANGCHAIN DOCOMUNENT OBJECT EXPANSION**


LangChain wraps every chunk of the text into a specific Python object called a Document, it has exactly two parts:


1.   page_content: A single string of text
2.   metadata: A standard Python Dictionary

RecursiveCharacterTextSplitter() slices the text into chunks of up to 1500 characters, maintaining a 200-character overlap to ensure no nuanced legal context is lost at the boundaries. It applies "contextual chunking" by prepending the exact legal identifier (such as [ARTICLE 10]) directly into the text of every single slice, this guarantees that the LLM always knows exactly which part of the law it is referencing. Then it packages each segment into a LangChain Document object and enriches it with structured metadata, including the category, legal unit, and a flag indicating if it is a continuation resulting in a robust, searchable dataset ready for embedding into the FAISS vector database.

In [36]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""]
)

final_documents = []


for _, row in df_units.iterrows():
    unit_id = row['id']
    category = row['category']
    text = row['text']

    chunks = text_splitter.split_text(text)

    for i, chunk in enumerate(chunks):
        enriched_content = f"[{unit_id}] {chunk}"

        combined_metadata = {
            "source": "EU_AI_Act_2024",
            "category": category,
            "legal_unit": unit_id,
            "chunk_index": i,
            "is_continuation": i > 0
        }

        spacy_doc = nlp(enriched_content)
        extracted = defaultdict(set)

        for ent in spacy_doc.ents:
            if ent.label_ != "IGNORE":
                extracted[ent.label_].add(ent.text)

        for label, entity_set in extracted.items():
            combined_metadata[f"ner_{label}"] = list(entity_set)

        doc = Document(
            page_content=enriched_content,
            metadata=combined_metadata
        )

        final_documents.append(doc)

print(f"Total vectorized chunks: {len(final_documents)}")

Total vectorized chunks: 577


Check of the information a single chunk is holding:

In [37]:
sample_doc = random.choice(final_documents)

for key, value in sample_doc.metadata.items():
    if not key.startswith("ner_"):
        clean_key = key.replace('_', ' ').title()
        print(f"{clean_key}: {value}")


found_entities = False
for key, value in sample_doc.metadata.items():
    if key.startswith("ner_"):
        found_entities = True
        print(f"{key}: {value}")

if not found_entities:
    print("(No entities found in this specific chunk)")

print("\n--- Text Preview ---")
print(f"{sample_doc.page_content[:800]}...")

Source: EU_AI_Act_2024
Category: ANNEX
Legal Unit: ANNEX VII
Chunk Index: 3
Is Continuation: True
ner_DOCUMENT: ['ANNEX VII']
ner_LEGAL_ROLE: ['provider']
ner_RISK_LEVEL: ['high-risk']
ner_LAW: ['Section 2', 'Chapter III']
ner_GPE: ['Union']
ner_CARDINAL: ['4.6']

--- Text Preview ---
[ANNEX VII] . Where the notified body is not satisfied with the tests carried out by the provider, the notified body shall itself directly carry out adequate tests, as appropriate. 4.5. Where necessary to assess the conformity of the high-risk AI system with the requirements set out in Chapter III, Section 2, after all other reasonable means to verify conformity have been exhausted and have proven to be insufficient, and upon a reasoned request, the notified body shall also be granted access to the training and trained models of the AI system, including its relevant parameters. Such access shall be subject to existing Union law on the protection of intellectual property and trade secrets. 4.6. The decisio

An additional refinement step was implemented to optimize the dataset for the Retrieval-Augmented Generation architecture. Following the initial text splitting, a filtering mechanism was applied to remove 7 low-value, boilerplate chunks, specifically legislative preambles and fragmented text containing fewer than 100 characters. Addiotionaly, leading punctuation artifacts generated by the recursive splitting process were stripped. This reduction from 577 to 570 highly concentrated text chunks ensures the FAISS vector database is populated exclusively with semantically dense legal provisions. By eliminating contextual noise, this approach maximizes embedding precision, reduces retrieval latency, and mitigates the risk of LLM hallucinations during the generation phase.

In [38]:
polished_documents = []
deleted_chunks = []

for doc in final_documents:
    clean_text = re.sub(r"^(\[.*?\])\s*[,.]+\s*", r"\1 ", doc.page_content)

    if len(clean_text) > 100:
        doc.page_content = clean_text
        polished_documents.append(doc)
    else:
        doc.page_content = clean_text
        deleted_chunks.append(doc)

print(f"Original chunks: {len(final_documents)}")
print(f"Polished chunks ready for vectorization: {len(polished_documents)}")
print(f"Removed {len(deleted_chunks)} low-value chunks.\n")

Original chunks: 577
Polished chunks ready for vectorization: 570
Removed 7 low-value chunks.



In [39]:
import json
import os

save_path = "data"

if not os.path.exists(save_path):
    os.makedirs(save_path)

polished_data = [
    {
        "page_content": doc.page_content,
        "metadata": doc.metadata
    }
    for doc in polished_documents
]

file_path = os.path.join(save_path, "polished_chunks.json")

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(polished_data, f, ensure_ascii=False, indent=2)

print(f"Chunks successfully saved to {file_path}")

Chunks successfully saved to data/polished_chunks.json


In [40]:
print(" DELETED CHUNKS PREVIEW ")
if not deleted_chunks:
    print("No chunks were deleted.")
else:
    for i, doc in enumerate(deleted_chunks):
        print(f"\n--- Deleted Chunk {i+1} ---")
        print(f"Legal Unit: {doc.metadata.get('legal_unit', 'Unknown')}")
        print(f"Length: {len(doc.page_content)} characters")
        print(f"Text: '{doc.page_content}'")

 DELETED CHUNKS PREVIEW 

--- Deleted Chunk 1 ---
Legal Unit: RECITAL 8
Length: 12 characters
Text: '[RECITAL 8] '

--- Deleted Chunk 2 ---
Legal Unit: RECITAL 35
Length: 31 characters
Text: '[RECITAL 35] Such data includes'

--- Deleted Chunk 3 ---
Legal Unit: ARTICLE 3
Length: 12 characters
Text: '[ARTICLE 3] '

--- Deleted Chunk 4 ---
Legal Unit: ARTICLE 5
Length: 37 characters
Text: '[ARTICLE 5] Prohibited AI practices 1'

--- Deleted Chunk 5 ---
Legal Unit: ARTICLE 13
Length: 13 characters
Text: '[ARTICLE 13] '

--- Deleted Chunk 6 ---
Legal Unit: ARTICLE 16
Length: 13 characters
Text: '[ARTICLE 16] '

--- Deleted Chunk 7 ---
Legal Unit: ARTICLE 66
Length: 13 characters
Text: '[ARTICLE 66] '


Running displacy one final time on the polished RAG chunks to prove that the custom entities defined in PATTERNS survived the text splitting, metadata injection, and string cleaning processes without being corrupted, and showcasing the accuracy reached by the ML model to identify entity from the AI act.

In [41]:
sample_chunks = random.sample(polished_documents, 3)

for i, doc in enumerate(sample_chunks):
    print(f"\n--- CHUNK {i+1} (Source: {doc.metadata['legal_unit']}) ---")
    spacy_doc = nlp(doc.page_content)

    custom_colors = {
        "LEGAL_ROLE": "linear-gradient(90deg, #f1c40f, #f39c12)",
        "RISK_LEVEL": "linear-gradient(90deg, #e74c3c, #c0392b)",
        "CONCEPT": "linear-gradient(90deg, #3498db, #2980b9)",
        "DOCUMENT": "linear-gradient(90deg, #9b59b6, #8e44ad)",
        "LAW": "linear-gradient(90deg, #34495e, #2c3e50)",
        "GPE": "linear-gradient(90deg, #2ecc71, #27ae60)"
    }

    #  hides the 'IGNORE' tags
    visible_ents = ["ORG", "GPE", "DOCUMENT", "LAW", "CONCEPT", "LEGAL_ROLE", "RISK_LEVEL"]

    options = {
        "colors": custom_colors,
        "ents": visible_ents
    }

    displacy.render(spacy_doc, style="ent", jupyter=True, options=options)


--- CHUNK 1 (Source: RECITAL 59) ---



--- CHUNK 2 (Source: ARTICLE 66) ---



--- CHUNK 3 (Source: RECITAL 22) ---


**ENTITY TYPE DISTRIBUTION**


Not all extracted metadata provide the same level of value for retrieval and reasoning. Some fields are relatively generic or structurally redundant and may have limited impact on ranking performance, while others are particularly informative in a legal and regulatory domain. In particular, risk levels, explicit legal references (e.g. articles and paragraphs), and regulatory actors carry strong semantic and normative signals and can significantly improve filtering, disambiguation, and answer grounding in a RAG system. Nevertheless, all extracted metadata are retained, as they may become useful for future extensions.

In [42]:
all_entity_counts = Counter()

for doc in polished_documents:
    for key, val_list in doc.metadata.items():
        if key.startswith("ner_"):
            clean_label = key.replace("ner_", "")
            all_entity_counts[clean_label] += len(val_list)

label_df = pd.DataFrame(all_entity_counts.most_common(), columns=['Entity Type', 'Count'])

fig = px.bar(
    label_df,
    x='Entity Type',
    y='Count',
    title='Named Entity Type Distribution (All Entities)',
    color='Count',
    color_continuous_scale='RdBu_r',
    text_auto=True
)

fig.update_layout(
    xaxis_title='Entity Type',
    yaxis_title='Frequency',
    xaxis_tickangle=-45,
    template='plotly_white',
    showlegend=False
)

fig.show()

Finally the chunks are ready and is possible to check the final lenght distribution.

In [43]:
chunk_lengths = [len(doc.page_content) for doc in polished_documents]
avg_len = sum(chunk_lengths) / len(chunk_lengths) if chunk_lengths else 0

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=chunk_lengths,
    nbinsx=50,
    marker_color='#e74c3c',
    marker_line=dict(color='white', width=1),
    name='Chunk Lengths'
))

fig.add_vline(x=1500, line_dash="dash", line_color="#c0392b")

fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    line=dict(color='#c0392b', dash='dash'),
    name='Max Chunk Limit (1500)'
))

fig.add_vline(x=avg_len, line_dash="solid", line_color="#f39c12")


fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='lines',
    line=dict(color='#f39c12'),
    name=f'Average Length ({avg_len:.0f})'
))

fig.update_layout(
    title='<b>Distribution of Final Vectorized Chunk Lengths</b>',
    xaxis_title="Characters per Chunk",
    yaxis_title="Number of Chunks",
    template='plotly_white',
    bargap=0.05,
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.5)"
    ),
    height=500
)


fig.show()

Below is shown a barplot of the absolute count of chunks belonging to RECITAL, ARTICLE, and ANNEX. This proves that the dataset is heavily weighted toward operative legal text (Articles) but retains the contextual intent (Recitals).

In [44]:
categories = [doc.metadata['category'] for doc in polished_documents]
category_counts = Counter(categories)
sorted_categories = sorted(category_counts.items(), key=lambda x: x[1], reverse=True)

labels = [x[0] for x in sorted_categories]
values = [x[1] for x in sorted_categories]

fig = go.Figure(data=[
    go.Bar(
        x=labels,
        y=values,
        text=values,
        textposition='auto',
        marker=dict(
            color=values,
            colorscale=[
                [0, '#ffcccc'],
                [1, '#b30000']
            ],
            showscale=False,
            line=dict(color='white', width=1.5)
        )
    )
])

fig.update_layout(
    title='<b>Document Structural Balance</b><br><sup>Chunks per Category</sup>',
    xaxis_title="Category",
    yaxis_title="Number of Vectorized Chunks",
    template='plotly_white',
    font=dict(family="Arial", size=12),
    bargap=0.2,
    height=500
)


fig.show()

# TF-IDF EDA

The TF-IDF approach is an essential exploratory tool because it goes far beyond simple word counts, which can often be misleading. TF-IDF evaluates the importance of a word by balancing its local frequency within a specific chunk against its global rarity across the entire dataset.  It deliberately penalizes common words that appear everywhere and assigns greater weight to terms that uniquely characterize individual documents. This mathematical filtering ensures that words with higher TF-IDF scores genuinely represent the most significant and meaningful concepts of the text.

In [45]:
corpus = [doc.page_content for doc in polished_documents]

custom_stop_words = ['shall', 'article', 'regulation', 'paragraph', 'annex',
                     'member', 'states', 'union', 'eu', 'referred', 'accordance', 'chapter', 'recital']

all_stop_words = list(ENGLISH_STOP_WORDS) + custom_stop_words

vectorizer = TfidfVectorizer(stop_words=all_stop_words, max_features=1000)
tfidf_matrix = vectorizer.fit_transform(corpus)

word_scores = tfidf_matrix.sum(axis=0).A1
words = vectorizer.get_feature_names_out()

df_words = pd.DataFrame({'word': words, 'score': word_scores})
top_20_words = df_words.sort_values(by='score', ascending=False).head(20)

top_20_words = top_20_words.sort_values(by='score', ascending=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    y=top_20_words['word'],
    x=top_20_words['score'],
    orientation='h',
    marker=dict(
        color='#9b59b6',
        line=dict(color='white', width=1)
    ),
    name='TF-IDF Score'
))

fig.update_layout(
    title='<b>Top 20 Most Important Terms in the AI Act (TF-IDF)</b>',
    xaxis_title="Cumulative TF-IDF Score",
    yaxis_title="Terms",
    template='plotly_white',
    height=700,
    yaxis={'categoryorder':'total ascending'},
    margin=dict(l=150)
)

fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='rgba(0,0,0,0.1)')

fig.show()

# EMBEDDDING THE CHUNKS AND CREATION OF DATABASE

**BGE_M3**


It uses a shared vector space for over 100 languages. This is critical for the AI Act, as it allows a user to query in their native tongue and retrieve the precise corresponding legal section, even if the primary knowledge base is in English, BGE-M3 demonstrates state-of-the-art cross-lingual transfer.
By setting normalize_embeddings=True, the model's ability to weight important legal keywords in enhanced. In the context of the AI Act, this means the model "learns" that terms like "High-Risk" are more vital to the search result than common stop-words

In [57]:
hf_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)
print("Model loaded")

print(f"Embedding {len(polished_documents)} chunks. This will take a moment...")

vectorstore = FAISS.from_documents(polished_documents, hf_embeddings) # text to gpu, vectorizes, and organizes in FAISS index
print("FAISS index built")

vectorstore.save_local(local_persistence_path)

print("\n Vector database created and saved")

/tmp/ipykernel_17100/174533229.py:1: LangChainDeprecationWarning:

The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Model loaded
Embedding 570 chunks. This will take a moment...
FAISS index built

 Vector database created and saved


# RETRIVAL LOGIC

This search architecture uses LangChain’s EnsembleRetriever to combine the strengths of semantic and keyword-based retrieval methods. While standard FAISS vector search (dense retrieval) excels at understanding the contextual meaning of queries, it can sometimes miss exact matches, such as specific legal terms or article numbers. On the other hand, the BM25 algorithm (sparse retrieval) is highly precise for keyword matching but lacks semantic understanding.

By combining both approaches with equal influence (weights=[0.5, 0.5]), the system executes both searches in parallel and merges their outputs using Reciprocal Rank Fusion (RRF). This mathematical algorithm recalculates the score of each document based on its rank position in both individual lists. Consequently, legal chunks that contain both the exact terminology (captured by BM25) and the correct contextual meaning (captured by FAISS) are elevated to the top of the final results. This dual-engine approach drastically increases overall retrieval precision, ensuring the generative AI model is grounded in the most accurate and comprehensive legal context possible.

In [58]:
# instanciating the model and loading database
hf_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)


vectorstore = FAISS.load_local(
    folder_path=local_persistence_path,
    embeddings=hf_embeddings,
    allow_dangerous_deserialization=True
)
print("Database loaded")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Database loaded


In [59]:
#  INITIALIZE RETRIEVAL ENGINE

# Find all NER keys
all_possible_ner_keys = set()
for doc in polished_documents:
    for key in doc.metadata.keys():
        if key.startswith("ner_"):
            all_possible_ner_keys.add(key)
all_possible_ner_keys = sorted(list(all_possible_ner_keys))

# Initialize the individual retrievers
bm25_retriever = BM25Retriever.from_documents(polished_documents)
bm25_retriever.k = 3
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Combine them into the hybrid engine
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5] # 50% keyword, 50% semantic
)

print("✅ Engine ready! (all_possible_ner_keys and hybrid_retriever are loaded into memory)")

✅ Engine ready! (all_possible_ner_keys and hybrid_retriever are loaded into memory)


**HYBRID RETRIVAL (METADATA-FILTERED)**

Standard search relies strictly on mathematical scoring (vector proximity and keyword frequency) to rank documents. While highly effective at finding general contextual relevance, it cannot guarantee that the retrieved documents adhere to specific factual constraints, such as ensuring a legal obligation strictly applies to a "deployer" rather than a "provider".
To solve this, the code below broadens the initial search space by retrieving the top 15 results from both the dense and sparse retrievers. Once the Reciprocal Rank Fusion algorithm merges and mathematically ranks these chunks into a list, the loop evaluates the pre-extracted Named Entity Recognition tags attached to each chunk's metadata. Any chunk that does not explicitly contain the target entity ("deployer") is immediately discarded, regardless of how high its mathematical search score was.
This architecture increases precision by combining the semantic intelligence of a standard search with the rigid, rules-based logic of metadata filtering. It eliminates the risk of "entity confusion" ensuring the generative LLM only reads legal rules that definitively apply to the exact stakeholder requested in the prompt.

In [60]:
query = "What are the registration obligations for high-risk biometric systems?"
print(f"User question: {query}\n")

bm25_retriever.k = 15
faiss_retriever.search_kwargs = {"k": 15}

raw_hybrid_chunks = hybrid_retriever.invoke(query)

filtered_chunks = []
for chunk in raw_hybrid_chunks:
    legal_roles = chunk.metadata.get("ner_LEGAL_ROLE", [])

    if any("deployer" in role.lower() for role in legal_roles):
        filtered_chunks.append(chunk)

if not filtered_chunks:
    print("No chunks found with 'deployer' metadata.")
else:
    for i, chunk in enumerate(filtered_chunks):
        print(f"FILTERED HYBRID RESULT {i+1}")
        print("-" * 60)

        # standard metadata
        print("GENERAL METADATA:")
        for key, value in chunk.metadata.items():
            if not key.startswith("ner_"):
                clean_key = key.replace('_', ' ').title()
                print(f"  • {clean_key}: {value}")

        # 2. Print NER metadata
        print("\nEXTRACTED ENTITIES:")
        has_entities = False

        for ner_key in all_possible_ner_keys:
            display_name = ner_key.replace('ner_', '').upper()
            extracted_entities = chunk.metadata.get(ner_key, [])

            if extracted_entities:
                if ner_key == "ner_LEGAL_ROLE":
                    print(f"  • {display_name} (TARGET FOUND): {extracted_entities}")
                else:
                    print(f"  • {display_name}: {extracted_entities}")
                has_entities = True

        if not has_entities:
            print("  • No entities detected.")

        print("\nTEXT PREVIEW:")
        print(f"{chunk.page_content[:500]}...\n")
        print("=" * 60 + "\n")

bm25_retriever.k = 3
faiss_retriever.search_kwargs = {"k": 3}

User question: What are the registration obligations for high-risk biometric systems?

FILTERED HYBRID RESULT 1
------------------------------------------------------------
GENERAL METADATA:
  • Source: EU_AI_Act_2024
  • Category: ARTICLE
  • Legal Unit: ARTICLE 26
  • Chunk Index: 3
  • Is Continuation: True

EXTRACTED ENTITIES:
  • CARDINAL: ['10', '2016/679', '8.', '2016/680', '9']
  • DOCUMENT: ['Directive', 'Regulation']
  • GPE: ['EU', 'Union']
  • LAW: ['ARTICLE 26', 'Article 35', 'Article 49', 'Article 13', 'Article 27', 'Article 71']
  • LEGAL_ROLE (TARGET FOUND): ['provider', 'deployers', 'deployer', 'Deployers']
  • RISK_LEVEL: ['high-risk']
  • TIME: ['48 hours']

TEXT PREVIEW:
[ARTICLE 26] 8. Deployers of high-risk AI systems that are public authorities, or Union institutions, bodies, offices or agencies shall comply with the registration obligations referred to in Article 49. When such deployers find that the high-risk AI system that they envisage using has not been regi

**Robust Retrieval and Filtering Function**

In [85]:
from typing import List, Optional

def retrieve_and_filter_chunks(
    query: str,
    hybrid_retriever,
    target_role: Optional[str] = None,
    target_risk: Optional[str] = None,
    fetch_k: int = 60,
    final_k: int = 5
) -> List:
    """
    Retrieves chunks using hybrid search and filters them using soft metadata matching.
    """
    for retriever in hybrid_retriever.retrievers:
        if hasattr(retriever, 'k'):
            retriever.k = fetch_k
        elif hasattr(retriever, 'search_kwargs'):
            retriever.search_kwargs["k"] = fetch_k

    raw_chunks = hybrid_retriever.invoke(query)

    clean_role = target_role.lower().strip() if target_role else None
    clean_risk = target_risk.lower().replace("_", " ").replace("-", " ") if target_risk else None

    scored_chunks = []

    for chunk in raw_chunks:
        roles = [r.lower() for r in chunk.metadata.get("ner_LEGAL_ROLE", [])]
        risks = [r.lower().replace("_", " ").replace("-", " ") for r in chunk.metadata.get("ner_RISK_LEVEL", [])]

        score = 0

        if clean_role and any(clean_role in r for r in roles):
            score += 1

        if clean_risk and any(clean_risk in r for r in risks):
            score += 1
        if (not clean_role and not clean_risk) or score > 0:
            scored_chunks.append((score, chunk))

    scored_chunks.sort(key=lambda x: x[0], reverse=True)
    filtered_chunks = [chunk for score, chunk in scored_chunks]

    if len(filtered_chunks) < final_k:
        print(f"Warning: Only found {len(filtered_chunks)} metadata matches. Padding with semantic matches.")
        for chunk in raw_chunks:
            if chunk not in filtered_chunks:
                filtered_chunks.append(chunk)
            if len(filtered_chunks) >= final_k:
                break

    return filtered_chunks[:final_k]

In [62]:
from langchain_core.prompts import PromptTemplate

def build_super_prompt(user_role: Optional[str], user_risk: Optional[str]) -> PromptTemplate:
    #  Prepare the strings first
    clean_risk = user_risk.replace('_', ' ').upper() if user_risk else "GENERAL"
    clean_role = user_role.upper() if user_role else "STAKEHOLDER"

    if user_role or user_risk:
        profile_desc = f"USER CATEGORY: {clean_risk}\nUSER ROLE: {clean_role}"
        mismatch_instruction = (
            f"STRICT HIERARCHY: Your primary filter is '{clean_risk}'. "
            f"1. Scan the context. Does it mention rules for '{clean_risk}'? "
            f"2. If the context ONLY mentions 'HIGH-RISK' and the user is '{clean_risk}', "
            "you MUST state that the obligations for High-Risk do NOT apply to them."
        )
    else:
        profile_desc = "USER CATEGORY: NOT SPECIFIED\nUSER ROLE: NOT SPECIFIED"
        mismatch_instruction = "Provide a general overview across all mentioned risk levels."

    template = f"""### SYSTEM ROLE:
You are an expert Legal Advisor for the EU AI Act. You provide precise, grounded, and safe legal analysis.

### USER PROFILE (DO NOT IGNORE):
{profile_desc}

### GENERATION PROTOCOL:
1. **LEGAL VERIFICATION STEP:** {mismatch_instruction}
2. **NO HALLUCINATION:** Use ONLY the provided 'Retrieved Legal Context'. If the information is not there, say: "The provided excerpts do not contain information applicable to your specific profile."
3. **STRUCTURE:** - Start with a direct answer regarding the user's specific status ({clean_risk}).
   - Provide a professional explanation with in-text citations like [Article 5] or [Recital 10].
   - If the context discusses a different risk level than the user's, explicitly explain the difference.
4. **SOURCES:** At the very end, add a horizontal line '---' followed by a bulleted list of the Articles/Recitals used.

### RETRIEVED LEGAL CONTEXT:
{{context}}

### USER QUESTION:
{{question}}

### FINAL ANALYSIS:
As a {clean_risk} {clean_role} under the AI Act, here is the analysis of your obligations:
"""
    return PromptTemplate(
        input_variables=["context", "question"],
        template=template
    )

A text-generation pipeline is configured with specific hyperparameters to control output quality. By setting the temperature to 0.1, the model is forced toward "greedy decoding," prioritizing factual accuracy and consistency over creative variation.

LangChain Integration: The native Hugging Face pipeline is wrapped in a HuggingFacePipeline object. This abstraction allows the local model to be easily integrated into broader LangChain workflows, such as Retrieval-Augmented Generation (RAG), while the generate_final_answer function provides a clean interface for invoking the model and retrieving stripped, text-only responses.

In [63]:
pip install langchain-huggingface accelerate

In [64]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('hf_gemma')
login(token=hf_token)


model_id = "google/gemma-2-2b-it"
print(f"Loading {model_id}...")

llm_tokenizer = AutoTokenizer.from_pretrained(model_id)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

# also try: temperature=0.01, repetition_penalty=1.2,do_sample=False
text_generation_pipeline = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    max_length=None,
    return_full_text=False
)

hf_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

def generate_final_answer(final_prompt: str) -> str:
    print("Generating answer... (this might take a few seconds)")
    response = hf_llm.invoke(final_prompt)
    return response.strip()

Loading google/gemma-2-2b-it...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_length', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


This code segment represents the core logic of a Risk-Aware RAG  system designed for regulatory compliance. It essentially acts as an automated "AI Act Advisor" by first classifying a user's business idea and then tailoring a legal response based on that risk level.

The system uses a fine-tuned DistilBERT model to categorize the user’s startup idea into one of four risk tiers (Minimal, Limited, High, or Unacceptable). If the model detects "Unacceptable Risk," the system halts the generation process entirely. This prevents the LLM from providing "how-to" advice for systems that are legally prohibited under the EU AI Act.

For legal ideas, the code uses an EnsembleRetriever to find relevant regulatory documentation. This combines two distinct search methods: BM25 (Keyword) and FAISS (Semantic).

Instead of a generic search, the retrieve_and_filter_chunks function filters data specifically for the user’s Role and Risk Level.

The system injects the retrieved legal chunks into a template. This ensures the LLM generates an answer grounded strictly in the provided regulatory text (reducing hallucinations) and tailored to the specific obligations of that risk category.

# DATASET FOR CLASSIFICATION

The following is the prompt used to generate the synthetic dataset used to train the following classifiers, with the help of this LLM: Gemini 3 (reasoning)

The prompt:
You are an expert AI data engineer and EU legal scholar. I am fine-tuning a DistilBERT text classifier and need a highly diverse synthetic dataset of AI product descriptions mapped to their official EU AI Act risk levels.
Generate exactly 100 rows of data in perfect CSV format.
The Rules:

The CSV must have exactly two columns: text and label.
The text column should be a 1 to 2 sentence description of a fictional AI system or startup product. Enclose every text description in double quotes ("").
The label column must be EXACTLY one of these four strings (no spaces, case-sensitive): Unacceptable_Risk, High_Risk, Limited_Risk, Minimal_Risk.
Generate exactly 25 examples for each of the 4 categories.
Crucial: Maximize industry diversity! Do not just use medical or HR examples. Invent systems for agriculture, toys, deepfakes, law enforcement, education, banking, video games, aviation, and chatbots.
Do not include any markdown formatting like ```csv or conversational text in your response. Output ONLY the raw CSV text starting with the header row, so I can copy and paste it directly.

In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.31.0 datasets evaluate accelerate -U

In [5]:
import pandas as pd
import os

file_path = "ai_risk_dataset.csv.txt"

if os.path.exists(file_path):
    print(f"Loading dataset from: {file_path}")
    df = pd.read_csv(file_path)

    print("\n--- FIRST 5 ROWS ---")
    print(df.head())

    print("\n--- LABEL DISTRIBUTION ---")
    print(df['label'].value_counts())
else:
    print(f"Error: '{file_path}' not found.")

Loading dataset from: ai_risk_dataset.csv.txt

--- FIRST 5 ROWS ---
                                                text              label
0  An AI-powered social scoring system used by mu...  Unacceptable_Risk
1  A real-time biometric identification system de...  Unacceptable_Risk
2  An AI application designed to subconsciously m...  Unacceptable_Risk
3  A predictive policing tool used to identify an...  Unacceptable_Risk
4  A cognitive-behavioral manipulation AI used by...  Unacceptable_Risk

--- LABEL DISTRIBUTION ---
label
Unacceptable_Risk    75
High_Risk            75
Limited_Risk         75
Minimal_Risk         75
Name: count, dtype: int64


# DISTILBERT CLASSIFIER

In [71]:
label_map = {"Minimal_Risk": 0, "Limited_Risk": 1, "High_Risk": 2, "Unacceptable_Risk": 3}
df['label'] = df['label'].map(label_map)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
eval_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train = train_dataset.map(tokenize_function, batched=True) # batched=True process faster
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    probabilities = softmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    recall = recall_score(labels, predictions, average='macro')

    auc = roc_auc_score(labels, probabilities, multi_class='ovr', average='macro')

    # nir
    largest_class_count = max(np.bincount(labels))
    nir = largest_class_count / len(labels)

    return {
        "accuracy": accuracy,
        "f1_macro": f1,
        "recall_macro": recall,
        "roc_auc": auc,
        "NIR": nir
    }

training_args = TrainingArguments(
    output_dir="./ai_risk_classifier",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [72]:
print(f"Train: {len(train_df)} | Val: {len(eval_df)} | Test: {len(test_df)}")

Train: 240 | Val: 30 | Test: 30


In [73]:
import os

trainer.train()

print("\n--- TEST SET EVALUATION ---")
test_results = trainer.predict(tokenized_test)

if test_results.metrics:
    for key, value in test_results.metrics.items():
        print(f"  • {key}: {value:.4f}")

save_path = "distilBERT_classifier"


trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"\n Training completed. Architecture and tokenizer saved locally to: {save_path}")

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Recall Macro,Roc Auc,Nir
1,1.323802,1.145700,0.800000,0.778105,0.790179,0.917146,0.266667
2,0.899882,0.649487,0.900000,0.887374,0.892857,0.993921,0.266667
3,0.473828,0.406561,0.833333,0.824208,0.830357,1.000000,0.266667
4,0.227983,0.235769,0.966667,0.964103,0.964286,1.000000,0.266667
5,0.097113,0.117197,1.000000,1.000000,1.000000,1.000000,0.266667
6,0.047941,0.056725,1.000000,1.000000,1.000000,1.000000,0.266667
7,0.029188,0.036775,1.000000,1.000000,1.000000,1.000000,0.266667
8,0.022545,0.035447,1.000000,1.000000,1.000000,1.000000,0.266667
9,0.018780,0.033221,1.000000,1.000000,1.000000,1.000000,0.266667
10,0.017534,0.031257,1.000000,1.000000,1.000000,1.000000,0.266667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



--- TEST SET EVALUATION ---


  • test_loss: 0.3637
  • test_accuracy: 0.8667
  • test_f1_macro: 0.8683
  • test_recall_macro: 0.8661
  • test_roc_auc: 0.9820
  • test_NIR: 0.2667
  • test_runtime: 0.5377
  • test_samples_per_second: 55.7950
  • test_steps_per_second: 7.4390


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


 Training completed. Architecture and tokenizer saved locally to: distilBERT_classifier


In [74]:
log_history = trainer.state.log_history

metrics_per_epoch = {}

for entry in log_history:
    if "epoch" not in entry:
        continue

    epoch = round(entry["epoch"], 2)

    if epoch not in metrics_per_epoch:
        metrics_per_epoch[epoch] = {'Epoch': int(epoch)}

    if "loss" in entry and "eval_loss" not in entry:
        metrics_per_epoch[epoch]["Training Loss"] = entry["loss"]

    if "eval_loss" in entry:
        metrics_per_epoch[epoch]["Validation Loss"] = entry["eval_loss"]
        metrics_per_epoch[epoch]["Accuracy"] = entry.get("eval_accuracy")
        metrics_per_epoch[epoch]["F1 Macro"] = entry.get("eval_f1_macro")
        metrics_per_epoch[epoch]["Recall Macro"] = entry.get("eval_recall_macro")
        metrics_per_epoch[epoch]["Roc Auc"] = entry.get("eval_roc_auc")
        metrics_per_epoch[epoch]["Nir"] = entry.get("eval_NIR")

metrics_df = pd.DataFrame(list(metrics_per_epoch.values()))

metrics_df = metrics_df.dropna(subset=["Validation Loss"])

columns_order = [
    "Epoch", "Training Loss", "Validation Loss",
    "Accuracy", "F1 Macro", "Recall Macro", "Roc Auc", "Nir"
]
existing_columns = [col for col in columns_order if col in metrics_df.columns]
metrics_df = metrics_df[existing_columns]

print(metrics_df.to_string(index=False))

 Epoch  Training Loss  Validation Loss  Accuracy  F1 Macro  Recall Macro  Roc Auc      Nir
     1       1.323802         1.145700  0.800000  0.778105      0.790179 0.917146 0.266667
     2       0.899882         0.649487  0.900000  0.887374      0.892857 0.993921 0.266667
     3       0.473828         0.406561  0.833333  0.824208      0.830357 1.000000 0.266667
     4       0.227983         0.235769  0.966667  0.964103      0.964286 1.000000 0.266667
     5       0.097113         0.117197  1.000000  1.000000      1.000000 1.000000 0.266667
     6       0.047941         0.056725  1.000000  1.000000      1.000000 1.000000 0.266667
     7       0.029188         0.036775  1.000000  1.000000      1.000000 1.000000 0.266667
     8       0.022545         0.035447  1.000000  1.000000      1.000000 1.000000 0.266667
     9       0.018780         0.033221  1.000000  1.000000      1.000000 1.000000 0.266667
    10       0.017534         0.031257  1.000000  1.000000      1.000000 1.000000 0.266667

In [75]:
log_history = trainer.state.log_history
data = []

for entry in log_history:
    if "loss" in entry and "eval_loss" not in entry:
        data.append({
            'Epoch': entry["epoch"],
            'Loss': entry["loss"],
            'Dataset': 'Training Loss'
        })


    if "eval_loss" in entry:
        data.append({
            'Epoch': entry["epoch"],
            'Loss': entry["eval_loss"],
            'Dataset': 'Validation Loss'
        })

df_base = pd.DataFrame(data)

if df_base[df_base['Dataset'] == 'Training Loss'].empty:
    print(" WARNING: No Training Loss data found in logs.")


all_frames = sorted(df_base['Epoch'].unique())
animated_data = []

for current_frame in all_frames:
    df_snapshot = df_base[df_base['Epoch'] <= current_frame].copy()
    df_snapshot['Frame'] = current_frame
    animated_data.append(df_snapshot)

df_animated = pd.concat(animated_data, ignore_index=True)


fig = px.line(
    df_animated,
    x="Epoch",
    y="Loss",
    color="Dataset",
    animation_frame="Frame",
    animation_group="Dataset",
    markers=True,
    title="Evolution of Training vs. Validation Loss",
    template="plotly_white",
    color_discrete_map={"Training Loss": "blue", "Validation Loss": "orange"},
    range_x=[0, df_base['Epoch'].max() * 1.05],
    range_y=[0, df_base['Loss'].max() * 1.1]
)

fig.show()

Performance report of DISTILBERT:

In [76]:
print(f"Accuracy: {test_results.metrics['test_accuracy'] * 100:.2f}%")
print(f"F1 Score: {test_results.metrics['test_f1_macro']:.4f}")
print(f"Recall:   {test_results.metrics['test_recall_macro']:.4f}")
print(f"ROC AUC:  {test_results.metrics['test_roc_auc']:.4f}")
print(f"NIR:      {test_results.metrics['test_NIR'] * 100:.2f}%")

Accuracy: 86.67%
F1 Score: 0.8683
Recall:   0.8661
ROC AUC:  0.9820
NIR:      26.67%


# LEGAL-BERT CLASSIFIER

In [12]:
import pandas as pd
import os

file_path = "ai_risk_dataset.csv.txt"

if os.path.exists(file_path):
    print(f"Loading dataset from: {file_path}")
    df = pd.read_csv(file_path)

    print("\n--- FIRST 5 ROWS ---")
    print(df.head())

    print("\n--- LABEL DISTRIBUTION ---")
    print(df['label'].value_counts())
else:
    print(f"Error: '{file_path}' not found.")

Loading dataset from: ai_risk_dataset.csv.txt

--- FIRST 5 ROWS ---
                                                text              label
0  An AI-powered social scoring system used by mu...  Unacceptable_Risk
1  A real-time biometric identification system de...  Unacceptable_Risk
2  An AI application designed to subconsciously m...  Unacceptable_Risk
3  A predictive policing tool used to identify an...  Unacceptable_Risk
4  A cognitive-behavioral manipulation AI used by...  Unacceptable_Risk

--- LABEL DISTRIBUTION ---
label
Unacceptable_Risk    75
High_Risk            75
Limited_Risk         75
Minimal_Risk         75
Name: count, dtype: int64


In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from scipy.special import softmax

label_map = {"Minimal_Risk": 0, "Limited_Risk": 1, "High_Risk": 2, "Unacceptable_Risk": 3}
df['label'] = df['label'].map(label_map)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
eval_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)
test_dataset = Dataset.from_pandas(test_df)


model_checkpoint = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=4)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probabilities = softmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    recall = recall_score(labels, predictions, average='macro')
    auc = roc_auc_score(labels, probabilities, multi_class='ovr', average='macro')

    largest_class_count = max(np.bincount(labels))
    nir = largest_class_count / len(labels)

    return {
        "accuracy": accuracy,
        "f1_macro": f1,
        "recall_macro": recall,
        "roc_auc": auc,
        "NIR": nir
    }


training_args = TrainingArguments(
    output_dir="./legal_ai_risk_classifier",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were new

In [14]:
import os

trainer.train()

print("\n--- TEST SET EVALUATION ---")
test_results = trainer.predict(tokenized_test)

if test_results.metrics:
    for key, value in test_results.metrics.items():
        print(f"  • {key}: {value:.4f}")

save_path = "legalBERT_classifier"


trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"\n Training completed. Architecture and tokenizer saved locally to: {save_path}")

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Recall Macro,Roc Auc,Nir
1,1.370815,1.300663,0.400000,0.277174,0.406250,0.784347,0.266667
2,1.240285,1.002902,0.733333,0.725757,0.736607,0.955304,0.266667
3,0.912061,0.725229,0.800000,0.781944,0.794643,0.966006,0.266667
4,0.584639,0.592700,0.833333,0.809926,0.825893,0.972976,0.266667
5,0.438646,0.475157,0.933333,0.935417,0.937500,0.978429,0.266667
6,0.371792,0.464994,0.866667,0.856124,0.861607,0.982293,0.266667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


--- TEST SET EVALUATION ---


  • test_loss: 0.4634
  • test_accuracy: 0.8667
  • test_f1_macro: 0.8609
  • test_recall_macro: 0.8661
  • test_roc_auc: 0.9744
  • test_NIR: 0.2667
  • test_runtime: 0.9575
  • test_samples_per_second: 31.3300
  • test_steps_per_second: 4.1770


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


 Training completed. Architecture and tokenizer saved locally to: legalBERT_classifier


In [15]:
log_history = trainer.state.log_history

metrics_per_epoch = {}

for entry in log_history:
    if "epoch" not in entry:
        continue

    epoch = round(entry["epoch"], 2)

    if epoch not in metrics_per_epoch:
        metrics_per_epoch[epoch] = {'Epoch': int(epoch)}

    if "loss" in entry and "eval_loss" not in entry:
        metrics_per_epoch[epoch]["Training Loss"] = entry["loss"]

    if "eval_loss" in entry:
        metrics_per_epoch[epoch]["Validation Loss"] = entry["eval_loss"]
        metrics_per_epoch[epoch]["Accuracy"] = entry.get("eval_accuracy")
        metrics_per_epoch[epoch]["F1 Macro"] = entry.get("eval_f1_macro")
        metrics_per_epoch[epoch]["Recall Macro"] = entry.get("eval_recall_macro")
        metrics_per_epoch[epoch]["Roc Auc"] = entry.get("eval_roc_auc")
        metrics_per_epoch[epoch]["Nir"] = entry.get("eval_NIR")

metrics_df = pd.DataFrame(list(metrics_per_epoch.values()))

metrics_df = metrics_df.dropna(subset=["Validation Loss"])

columns_order = [
    "Epoch", "Training Loss", "Validation Loss",
    "Accuracy", "F1 Macro", "Recall Macro", "Roc Auc", "Nir"
]
existing_columns = [col for col in columns_order if col in metrics_df.columns]
metrics_df = metrics_df[existing_columns]

print(metrics_df.to_string(index=False))

 Epoch  Training Loss  Validation Loss  Accuracy  F1 Macro  Recall Macro  Roc Auc      Nir
     1       1.370815         1.300663  0.400000  0.277174      0.406250 0.784347 0.266667
     2       1.240285         1.002902  0.733333  0.725757      0.736607 0.955304 0.266667
     3       0.912061         0.725229  0.800000  0.781944      0.794643 0.966006 0.266667
     4       0.584639         0.592700  0.833333  0.809926      0.825893 0.972976 0.266667
     5       0.438646         0.475157  0.933333  0.935417      0.937500 0.978429 0.266667
     6       0.371792         0.464994  0.866667  0.856124      0.861607 0.982293 0.266667


In [16]:
log_history = trainer.state.log_history
data = []

for entry in log_history:
    if "loss" in entry and "eval_loss" not in entry:
        data.append({
            'Epoch': entry["epoch"],
            'Loss': entry["loss"],
            'Dataset': 'Training Loss'
        })


    if "eval_loss" in entry:
        data.append({
            'Epoch': entry["epoch"],
            'Loss': entry["eval_loss"],
            'Dataset': 'Validation Loss'
        })

df_base = pd.DataFrame(data)

if df_base[df_base['Dataset'] == 'Training Loss'].empty:
    print(" WARNING: No Training Loss data found in logs.")


all_frames = sorted(df_base['Epoch'].unique())
animated_data = []

for current_frame in all_frames:
    df_snapshot = df_base[df_base['Epoch'] <= current_frame].copy()
    df_snapshot['Frame'] = current_frame
    animated_data.append(df_snapshot)

df_animated = pd.concat(animated_data, ignore_index=True)


fig = px.line(
    df_animated,
    x="Epoch",
    y="Loss",
    color="Dataset",
    animation_frame="Frame",
    animation_group="Dataset",
    markers=True,
    title="Evolution of Training vs. Validation Loss",
    template="plotly_white",
    color_discrete_map={"Training Loss": "blue", "Validation Loss": "orange"},
    range_x=[0, df_base['Epoch'].max() * 1.05],
    range_y=[0, df_base['Loss'].max() * 1.1]
)

fig.show()

In [17]:
print(f"Accuracy: {test_results.metrics['test_accuracy'] * 100:.2f}%")
print(f"F1 Score: {test_results.metrics['test_f1_macro']:.4f}")
print(f"Recall:   {test_results.metrics['test_recall_macro']:.4f}")
print(f"ROC AUC:  {test_results.metrics['test_roc_auc']:.4f}")
print(f"NIR:      {test_results.metrics['test_NIR'] * 100:.2f}%")

Accuracy: 86.67%
F1 Score: 0.8609
Recall:   0.8661
ROC AUC:  0.9744
NIR:      26.67%


# RNN (GRU BASED)

Loading the data and using the same train, val, test split as the previus distilbert model to ensure consistency for the following performance comparison.

In [18]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

--2026-03-20 11:04:35--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-03-20 11:04:35--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-03-20 11:04:35--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [19]:
import pandas as pd
import os

file_path = "ai_risk_dataset.csv.txt"

if os.path.exists(file_path):
    print(f"Loading dataset from: {file_path}")
    df = pd.read_csv(file_path)
else:
    print(f"Error: '{file_path}' not found.")

Loading dataset from: ai_risk_dataset.csv.txt


In [20]:
df = pd.read_csv(file_path)
df['text'] = df['text'].str.replace('-', ' ', regex=False)

label_map = {"Minimal_Risk": 0, "Limited_Risk": 1, "High_Risk": 2, "Unacceptable_Risk": 3}
df['label'] = df['label'].map(label_map)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
eval_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])


X_train, y_train = train_df['text'].to_numpy(), train_df['label'].to_numpy()
X_eval, y_eval = eval_df['text'].to_numpy(), eval_df['label'].to_numpy()
X_test, y_test = test_df['text'].to_numpy(), test_df['label'].to_numpy()

In [21]:
embeddings_index = {}
with open('glove.6B.100d.txt', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32') # embddings
        embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} pre-trained word vectors.")

Found 400000 pre-trained word vectors.


In [22]:
from tensorflow.keras.layers import TextVectorization
max_features = 1500
sequence_length = 100

vectorizer = TextVectorization(
    max_tokens=max_features,
    output_sequence_length=sequence_length
)

vectorizer.adapt(X_train)

voc = vectorizer.get_vocabulary()
word_index = dict(zip(voc, range(len(voc))))
num_tokens = len(voc)
embedding_dim = 100 #  match the GloVe file downloaded (100d)

embedding_matrix = np.zeros((num_tokens, embedding_dim))

hits = 0
misses = 0
for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
        hits += 1
    else:
        misses += 1

print(f"Converted {hits} words ({misses} misses)")

Converted 1486 words (8 misses)


In [23]:
missing_words = []

for word in word_index.keys():
    if word not in embeddings_index:
        missing_words.append(word)

print("missing words:", missing_words)

missing words: ['', '[UNK]', np.str_('chatbot'), np.str_('deepfake'), np.str_('neurodivergent'), np.str_('librarys'), np.str_('gamified'), np.str_('deprioritizes')]


In [24]:
from tensorflow.keras.layers import SpatialDropout1D
model = Sequential([
    Input(shape=(1,), dtype=tf.string),
    vectorizer,

    Embedding(
        input_dim=num_tokens,
        output_dim=embedding_dim,
        embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
        trainable=False,
    ),

    SpatialDropout1D(0.2),

    Bidirectional(
        GRU(
            32,
            dropout=0.2,
            recurrent_dropout=0.1,
            return_sequences=False
        )
    ),

    Dense(32, activation='relu'),
    Dropout(0.3),

    Dense(4, activation='softmax')
])


total_steps = (len(X_train) // 8) * 10 # epochs = 10

# linear decay
lr_scheduler = PolynomialDecay(
    initial_learning_rate=1e-3,
    end_learning_rate=1e-5,
    decay_steps=total_steps,
    power=1.0
)


model.compile(
    optimizer=AdamW(learning_rate=lr_scheduler, weight_decay=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [25]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_eval, y_eval),
    epochs=10,
    batch_size=8,
    callbacks=callbacks
)

Epoch 1/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 29s 644ms/step - accuracy: 0.2833 - loss: 1.3852 - val_accuracy: 0.2667 - val_loss: 1.3600
Epoch 2/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 568ms/step - accuracy: 0.2708 - loss: 1.4024 - val_accuracy: 0.4000 - val_loss: 1.3550
Epoch 3/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 20s 568ms/step - accuracy: 0.2458 - loss: 1.3815 - val_accuracy: 0.3667 - val_loss: 1.3451
Epoch 4/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 20s 550ms/step - accuracy: 0.3375 - loss: 1.3546 - val_accuracy: 0.3667 - val_loss: 1.3337
Epoch 5/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 21s 556ms/step - accuracy: 0.3333 - loss: 1.3505 - val_accuracy: 0.3667 - val_loss: 1.3278
Epoch 6/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 577ms/step - accuracy: 0.4292 - loss: 1.3059 - val_accuracy: 0.2667 - val_loss: 1.3199
Epoch 7/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 20s 561ms/step - accuracy: 0.4000 - loss: 1.2862 - val_accuracy: 0.3667 - val_loss: 1.3025
Epoch 8/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 16s 551ms/step - accuracy: 0.4292 - loss: 1.2832 - val_accu

In [26]:
metrics_dict = history.history
metrics_df = pd.DataFrame(metrics_dict)
metrics_df.insert(0, 'Epoch', range(1, len(metrics_df) + 1))
print(metrics_df.to_string(index=False))

 Epoch  accuracy     loss  val_accuracy  val_loss
     1  0.283333 1.385235      0.266667  1.359979
     2  0.270833 1.402356      0.400000  1.355041
     3  0.245833 1.381456      0.366667  1.345133
     4  0.337500 1.354593      0.366667  1.333722
     5  0.333333 1.350486      0.366667  1.327776
     6  0.429167 1.305852      0.266667  1.319896
     7  0.400000 1.286175      0.366667  1.302542
     8  0.429167 1.283172      0.466667  1.293013
     9  0.479167 1.260003      0.400000  1.280682
    10  0.462500 1.247344      0.400000  1.273363


In [27]:
# The model.fit() process in Keras returns a history object.
# extract the training and validation losses from it:
train_loss = history.history['loss']
eval_loss = history.history['val_loss']

# Keras epochs are 0-indexed
epochs = [e + 1 for e in history.epoch]

data = []

for e, l in zip(epochs, train_loss):
    data.append({'Epoch': e, 'Loss': l, 'Dataset': 'Training Loss'})
for e, l in zip(epochs, eval_loss):
    data.append({'Epoch': e, 'Loss': l, 'Dataset': 'Validation Loss'})

df_base = pd.DataFrame(data)

animated_data = []
unique_epochs = sorted(list(set(epochs)))


for current_frame in unique_epochs:
    df_snapshot = df_base[df_base['Epoch'] <= current_frame].copy()
    df_snapshot['Frame'] = current_frame
    animated_data.append(df_snapshot)

df_animated = pd.concat(animated_data, ignore_index=True)

fig = px.line(
    df_animated,
    x="Epoch",
    y="Loss",
    color="Dataset",
    animation_frame="Frame",
    animation_group="Dataset",
    markers=True,
    range_x=[0, df_base['Epoch'].max() + 0.2],
    range_y=[df_base['Loss'].min() * 0.9, df_base['Loss'].max() * 1.1],
    title="Evolution of Training vs. Validation Loss (GRU Model)",
    template="plotly_white",
    color_discrete_map={"Training Loss": "blue", "Validation Loss": "orange"}
)

fig.show()

In [28]:
probabilities = model.predict(X_test)
predictions = np.argmax(probabilities, axis=-1)

accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions, average='macro')
recall = recall_score(y_test, predictions, average='macro')
auc = roc_auc_score(y_test, probabilities, multi_class='ovr', average='macro')

largest_class_count = max(np.bincount(y_test))
nir = largest_class_count / len(y_test)

results = {
    "accuracy": accuracy,
    "f1_macro": f1,
    "recall_macro": recall,
    "roc_auc": auc,
    "NIR": nir
}

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


In [29]:
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

accuracy: 0.7667
f1_macro: 0.7681
recall_macro: 0.7589
roc_auc: 0.9281
NIR: 0.2667


# ULMFIT CLASSIFIER

In [30]:
file_path = "ai_risk_dataset.csv.txt"

df = pd.read_csv(file_path)
df['text'] = df['text'].str.replace('-', ' ', regex=False)

label_map = {"Minimal_Risk": 0, "Limited_Risk": 1, "High_Risk": 2, "Unacceptable_Risk": 3}
df['label'] = df['label'].map(label_map)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
eval_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

This stage implements the second step of the Universal Language Model Fine-tuning (ULMFiT) architecture: Domain Adaptation. We initialize an AWD-LSTM (ASGD Weight-Dropped LSTM) model pre-trained on the WikiText-103 dataset. We then feed it the raw text of the EU AI Act without any classification labels (is_lm=True). The model is trained using a next-word prediction objective.

The pre-trained Wikipedia model understands general English grammar and syntax, but it lacks knowledge of specific legal and technical terms.

With only 300 labeled examples for the final classifier, training from scratch would lead to poor performance, since is well known that lstm models need large datasets.

Slanted Triangular Learning Rates (STLR): The fit_one_cycle method dictates that the learning rate quickly increases to a peak and then slowly decays. This allows the model to rapidly traverse the loss landscape to find a good optimal region, then fine-tune its weights precisely.

In [46]:
import pandas as pd
from fastai.text.all import *

# extract raw text
ai_act_texts = [doc.page_content for doc in polished_documents]
df_lm = pd.DataFrame({'text': ai_act_texts})

# is_lm=True tells fastai to set up the data for predicting the next word
dls_lm = TextDataLoaders.from_df(df_lm, text_col='text', is_lm=True, valid_pct=0.1)

print(f"Vocab size: {len(dls_lm.vocab)}")

Vocab size: 2288


language_model_learner() function automatically creates a pytorch model for next word prediction.


AWD-LSTM uses multiple different types of dropout simultaneously (dropping out random words, dropping out connections between RNN layers and more)

drop_mult=0.3 --> lowering the amount of dropout to $30\%$ of the default, to avoid throwing away too much of its pre-trained knowledge.

In [47]:
# initialize the Language Model, wiki weight automaticallty downloaded
learn_lm = language_model_learner(dls_lm, AWD_LSTM, drop_mult=0.3, metrics=[accuracy, Perplexity()]) # perplexity to monitor (=exp(loss))


# fit_one_cycle applies Slanted Triangular Learning Rates
learn_lm.fit_one_cycle(1, 2e-2) # Train only the head with high learning rate for 1 epoch
learn_lm.unfreeze() # unlock the whole model
learn_lm.fit_one_cycle(4, 2e-3) # Train the whole model on AI Act for 4 epochs


learn_lm.save_encoder('finetuned_ai_act_encoder')
print("Language Model fine-tuned and encoder saved")

<div><progress max="105067061" value="105070592"></progress> 100.00% [105070592/105067061 00:02&lt;00:00]</div>

epoch,train_loss,valid_loss,accuracy,perplexity,time
0,3.408798,2.976976,0.404905,19.628378,00:06


epoch,train_loss,valid_loss,accuracy,perplexity,time
0,2.677083,2.627967,0.458346,13.845592,00:05
1,2.423723,2.391522,0.495353,10.930119,00:05
2,2.167372,2.307568,0.512974,10.049952,00:05
3,1.941536,2.299759,0.514021,9.971783,00:06


Language Model fine-tuned and encoder saved


This is the core classification phase. We construct a new TextDataLoaders object for the classification task, enforcing the exact same training/validation split used in previous baselines. Crucially, we pass text_vocab=dls_lm.vocab to synchronize the vocabulary mapping. We then load the customized "encoder" (the weights from Chunk 1) and attach a new, randomly initialized classification head for the 4 risk categories. Training is conducted via Gradual Unfreezing and Discriminative Learning Rates.


Instead of training all layers at once, layers are unfrozen one by one starting from the top (the classification head) down to the bottom (the embeddings).


In the code slice(1e-3/(2.6**4), 1e-3), different layers of the network receive different learning rates. The deepest layers capture fundamental language rules and require very small updates. The final layers capture task-specific features and require larger updates. The denominator 2.6**4 is an empirical heuristic proven in the ULMFiT paper to be optimal for scaling learning rates across AWD-LSTM layers.


training and validation losses across multiple discrete training phases are captured and stitched into dataframe.
Because fastai's fit_one_cycle method resets the internal recorder after every cycle, relying on a single final history extraction only yields the final phase's metrics. Manually extending lists (all_train_losses.extend(...)) after every freeze/unfreeze state guarantees the extraction of the metrics across all epochs.

Instead of treating the AWD-LSTM as thousands of individual, separate neural layers, fastai bundles them into Layer Groups.

Group 1 (Bottom): Word Embeddings (understanding basic vocabulary).

Group 2 (Middle): Lower LSTM layers (understanding basic grammar).

Group 3 (Higher): Upper LSTM layers (understanding complex AI Act context).

Group 4 (Top): The Classification Head

In [48]:
import pandas as pd

# use row marked as false for trainign and true for validation
train_df['is_valid'] = False
eval_df['is_valid'] = True
df_class = pd.concat([train_df, eval_df])

# Create the DataLoaders
dls_clas = TextDataLoaders.from_df(
    df_class,
    text_col='text',
    label_col='label',
    valid_col='is_valid',
    text_vocab=dls_lm.vocab, # vocab from LM
    bs=16
)

# Initialize Classifier and load the encoder
learn_clas = text_classifier_learner(dls_clas, AWD_LSTM, drop_mult=0.5, metrics=accuracy)
learn_clas = learn_clas.load_encoder('finetuned_ai_act_encoder')

# store the history
all_train_losses = []
all_valid_losses = []

# Train Head
learn_clas.fit_one_cycle(1, 2e-2) # here group 1 2 3 are locked
all_train_losses.extend([val[0] for val in learn_clas.recorder.values])
all_valid_losses.extend([val[1] for val in learn_clas.recorder.values])


learn_clas.freeze_to(-2) # freeze everything but the last 2 groups
learn_clas.fit_one_cycle(1, slice(1e-2/(2.6**4), 1e-2)) # final learning rate = 1e-2 assigned to the head, initial lr is 1e-2/(2.6**4 assigned to the bottom layer
all_train_losses.extend([val[0] for val in learn_clas.recorder.values])
all_valid_losses.extend([val[1] for val in learn_clas.recorder.values])

learn_clas.freeze_to(-3)
learn_clas.fit_one_cycle(1, slice(5e-3/(2.6**4), 5e-3))
all_train_losses.extend([val[0] for val in learn_clas.recorder.values])
all_valid_losses.extend([val[1] for val in learn_clas.recorder.values])

#  unfreeze whole model
learn_clas.unfreeze()
learn_clas.fit_one_cycle(3, slice(1e-3/(2.6**4), 1e-3))
all_train_losses.extend([val[0] for val in learn_clas.recorder.values])
all_valid_losses.extend([val[1] for val in learn_clas.recorder.values])

epoch,train_loss,valid_loss,accuracy,time
0,1.267021,1.149787,0.466667,00:02


epoch,train_loss,valid_loss,accuracy,time
0,0.950878,0.962054,0.600000,00:02


epoch,train_loss,valid_loss,accuracy,time
0,0.757135,0.879497,0.600000,00:02


epoch,train_loss,valid_loss,accuracy,time
0,0.538509,0.732186,0.700000,00:01
1,0.487857,0.670797,0.800000,00:01
2,0.461177,0.660367,0.833333,00:02


In [49]:
epochs = [e + 1 for e in range(len(all_train_losses))]

data = []

for e, t_loss, v_loss in zip(epochs, all_train_losses, all_valid_losses):
    data.append({'Epoch': e, 'Loss': t_loss, 'Dataset': 'Training Loss'})
    data.append({'Epoch': e, 'Loss': v_loss, 'Dataset': 'Validation Loss'})

df_base = pd.DataFrame(data)

animated_data = []
unique_epochs = sorted(list(set(epochs)))

for current_frame in unique_epochs:
    df_snapshot = df_base[df_base['Epoch'] <= current_frame].copy()
    df_snapshot['Frame'] = current_frame
    animated_data.append(df_snapshot)

df_animated = pd.concat(animated_data, ignore_index=True)

fig = px.line(
    df_animated,
    x="Epoch",
    y="Loss",
    color="Dataset",
    animation_frame="Frame",
    animation_group="Dataset",
    markers=True,
    range_x=[0.5, df_base['Epoch'].max() + 0.5],
    range_y=[df_base['Loss'].min() * 0.9, df_base['Loss'].max() * 1.1],
    title="Evolution of Training vs. Validation Loss (ULMFiT Model)",
    template="plotly_white",
    color_discrete_map={"Training Loss": "blue", "Validation Loss": "orange"}
)

fig.show()

In [50]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
import numpy as np

# dataloader for the test
dl_test = learn_clas.dls.test_dl(test_df, with_labels=True)
probabilities, y_test_fastai = learn_clas.get_preds(dl=dl_test)

probs_np = probabilities.numpy()
preds_np = np.argmax(probs_np, axis=-1)
y_true_np = y_test_fastai.numpy()

accuracy = accuracy_score(y_true_np, preds_np)
f1 = f1_score(y_true_np, preds_np, average='macro')
recall = recall_score(y_true_np, preds_np, average='macro')

try:
    auc = roc_auc_score(y_true_np, probs_np, multi_class='ovr', average='macro')
except ValueError:
    auc = "N/A"

largest_class_count = max(np.bincount(y_true_np))
nir = largest_class_count / len(y_true_np)

results = {
    "accuracy": accuracy,
    "f1_macro": f1,
    "recall_macro": recall,
    "roc_auc": auc,
    "NIR": nir
}

for metric, value in results.items():
    if isinstance(value, float):
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: {value}")

accuracy: 0.9333
f1_macro: 0.9286
recall_macro: 0.9286
roc_auc: 0.9907
NIR: 0.2667


In [79]:
learn_clas.export('ulmfit_ai_risk_classifier.pkl')

# RANDOM FOREST CLASSIFIER

In [53]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import pandas as pd

file_path = "ai_risk_dataset.csv.txt"
df = pd.read_csv(file_path)

X_train, X_temp, y_train, y_temp = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

X_eval, X_test, y_eval, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1500, stop_words='english')),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

param_grid = {
    'classifier__n_estimators': [100, 200, 300],

    'classifier__max_depth': [10, 20, 30, None],

    'classifier__max_features': ['sqrt', 'log2', 0.15, 0.2]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("\n=== Grid Search Complete ===")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation F1-Macro: {grid_search.best_score_:.4f}\n")

best_rf_model = grid_search.best_estimator_
predictions = best_rf_model.predict(X_test)

print("=== Best Model Performance on Test Set ===")
print(classification_report(y_test, predictions))


=== Grid Search Complete ===
Best Parameters: {'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__n_estimators': 100}
Best Cross-Validation F1-Macro: 0.7475

=== Best Model Performance on Test Set ===
                   precision    recall  f1-score   support

        High_Risk       1.00      0.38      0.55         8
     Limited_Risk       0.83      0.71      0.77         7
     Minimal_Risk       0.44      0.88      0.58         8
Unacceptable_Risk       0.80      0.57      0.67         7

         accuracy                           0.63        30
        macro avg       0.77      0.63      0.64        30
     weighted avg       0.76      0.63      0.64        30



# SVM CLASSIFIER

In [54]:
file_path = "ai_risk_dataset.csv.txt"
df = pd.read_csv(file_path)


X_train, X_temp, y_train, y_temp = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

X_eval, X_test, y_eval, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

svm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('classifier', SVC(kernel='linear', class_weight='balanced', random_state=42, probability = True))
])

param_grid = {
    # different levels of strictness for the SVM
    'classifier__C': [0.1, 1, 10, 100],

    # different vocabulary sizes
    'tfidf__max_features': [1000, 1500, 2000],

    # Unigrams (1,1) vs Unigrams + Bigrams (1,2)
    'tfidf__ngram_range': [(1, 1), (1, 2)]
}

grid_search_svm = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)

grid_search_svm.fit(X_train, y_train)

print("\n=== SVM Grid Search Complete ===")
print(f"Best Parameters: {grid_search_svm.best_params_}")
print(f"Best Cross-Validation F1-Macro: {grid_search_svm.best_score_:.4f}\n")

best_svm_model = grid_search_svm.best_estimator_
predictions = best_svm_model.predict(X_test)

print("=== Best SVM Model Performance on Test Set ===")
print(classification_report(y_test, predictions))

Fitting 5 folds for each of 24 candidates, totalling 120 fits

=== SVM Grid Search Complete ===
Best Parameters: {'classifier__C': 10, 'tfidf__max_features': 1500, 'tfidf__ngram_range': (1, 1)}
Best Cross-Validation F1-Macro: 0.8787

=== Best SVM Model Performance on Test Set ===
                   precision    recall  f1-score   support

        High_Risk       0.86      0.75      0.80         8
     Limited_Risk       0.83      0.71      0.77         7
     Minimal_Risk       0.70      0.88      0.78         8
Unacceptable_Risk       0.71      0.71      0.71         7

         accuracy                           0.77        30
        macro avg       0.78      0.76      0.77        30
     weighted avg       0.78      0.77      0.77        30



# XGBOOST CLASSIFIER

In [55]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

file_path = "ai_risk_dataset.csv.txt"
df = pd.read_csv(file_path)

X_train, X_temp, y_train, y_temp = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

X_eval, X_test, y_eval, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)


xgb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('classifier', XGBClassifier(random_state=42))
])

param_grid = {
    # XGBoost  parameters
    'classifier__learning_rate': [0.05, 0.1, 0.2],
    'classifier__max_depth': [1,2,3],         # depth trees
    'classifier__n_estimators': [100, 200, 300],     # number sequential trees

    # TF-IDF parameters
    'tfidf__max_features': [1000, 1500, 2000],
    'tfidf__ngram_range': [(1, 1), (1, 2)]
}


grid_search_xgb = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)


grid_search_xgb.fit(X_train, y_train_encoded)


print("\n=== XGBoost Grid Search Complete ===")
print(f"Best Parameters: {grid_search_xgb.best_params_}")
print(f"Best Cross-Validation F1-Macro: {grid_search_xgb.best_score_:.4f}\n")


best_xgb_model = grid_search_xgb.best_estimator_
predictions_encoded = best_xgb_model.predict(X_test)

predictions = le.inverse_transform(predictions_encoded)

Fitting 5 folds for each of 162 candidates, totalling 810 fits

=== XGBoost Grid Search Complete ===
Best Parameters: {'classifier__learning_rate': 0.2, 'classifier__max_depth': 1, 'classifier__n_estimators': 200, 'tfidf__max_features': 1000, 'tfidf__ngram_range': (1, 2)}
Best Cross-Validation F1-Macro: 0.6355



In [56]:
print("=== Best XGBoost Model Performance on Test Set ===")
print(classification_report(y_test, predictions))

=== Best XGBoost Model Performance on Test Set ===
                   precision    recall  f1-score   support

        High_Risk       0.67      0.75      0.71         8
     Limited_Risk       1.00      0.57      0.73         7
     Minimal_Risk       0.58      0.88      0.70         8
Unacceptable_Risk       0.80      0.57      0.67         7

         accuracy                           0.70        30
        macro avg       0.76      0.69      0.70        30
     weighted avg       0.75      0.70      0.70        30



# FINAL COMPARISON OF CLASSIFIERS

In [65]:
import pandas as pd

# Updated with new evaluation metrics
df_compare = pd.DataFrame([
    {
        "Model": "DistilBERT (Pre-trained)",
        "Accuracy": 0.8333,
        "F1 Macro": 0.8305,
        "Recall Macro": 0.8348,
        "ROC AUC": 0.9807,
        "NIR": 0.2667
    },
    {
        "Model": "GRU",
        "Accuracy": 0.7667,
        "F1 Macro": 0.7681,
        "Recall Macro": 0.7589,
        "ROC AUC": 0.9281,
        "NIR": 0.2667
    },
    {
        "Model": "Random Forest (Tuned)",
        "Accuracy": 0.6333,
        "F1 Macro": 0.6400,
        "Recall Macro": 0.6300,
        "ROC AUC": 0.6265, # Retained from previous report
        "NIR": 0.2667
    },
    {
        "Model": "Linear SVM (Tuned)",
        "Accuracy": 0.7667,
        "F1 Macro": 0.7700,
        "Recall Macro": 0.7600,
        "ROC AUC": 0.6047, # Retained from previous report
        "NIR": 0.2667
    },
    {
        "Model": "XGBoost (Tuned)",
        "Accuracy": 0.7000,
        "F1 Macro": 0.7000,
        "Recall Macro": 0.6900,
        "ROC AUC": 0.5710, # Retained from previous report
        "NIR": 0.2667
    },
    {
        "Model": "legalBERT (pretrained)",
        "Accuracy": 0.8667,
        "F1 Macro": 0.8609,
        "Recall Macro": 0.8661,
        "ROC AUC": 0.9744,
        "NIR": 0.2667
    },
    {
        "Model": "ULMFIT",
        "Accuracy": 0.9333,
        "F1 Macro": 0.9286,
        "Recall Macro": 0.9286,
        "ROC AUC": 0.9907,
        "NIR": 0.2667
    }
])

cols = ["Model", "Accuracy", "F1 Macro", "Recall Macro", "ROC AUC", "NIR"]
df_compare = df_compare[cols]

format_dict = {
    "Accuracy": "{:.2%}",
    "F1 Macro": "{:.4f}",
    "Recall Macro": "{:.4f}",
    "ROC AUC": "{:.4f}",
    "NIR": "{:.2%}"
}

display(df_compare.style.format(format_dict))

,Model,Accuracy,F1 Macro,Recall Macro,ROC AUC,NIR
0,DistilBERT (Pre-trained),83.33%,0.8305,0.8348,0.9807,26.67%
1,GRU,76.67%,0.7681,0.7589,0.9281,26.67%
2,Random Forest (Tuned),63.33%,0.6400,0.6300,0.6265,26.67%
3,Linear SVM (Tuned),76.67%,0.7700,0.7600,0.6047,26.67%
4,XGBoost (Tuned),70.00%,0.7000,0.6900,0.5710,26.67%
5,legalBERT (pretrained),86.67%,0.8609,0.8661,0.9744,26.67%
6,ULMFIT,93.33%,0.9286,0.9286,0.9907,26.67%


# GENERARTION

In [90]:
from fastai.text.all import load_learner

print("Loading ULMFiT model...")
ulmfit_model = load_learner("ulmfit_ai_risk_classifier.pkl")


reverse_label_map = {0: "Minimal_Risk", 1: "Limited_Risk", 2: "High_Risk", 3: "Unacceptable_Risk"}

bm25_retriever = BM25Retriever.from_documents(polished_documents)
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5] # 50% keyword, 50% semantic
)



user_startup_idea = "We are deploying an emotion recognition AI system in corporate workplaces and universities to constantly monitor the facial expressions of employees and students to infer their current emotional state and engagement levels."
user_question = "What are the CE marking and registration requirements for this biometric system?"

print("\n--- 1. PROCESSING USER INPUT ---")
print(f"Startup Idea: '{user_startup_idea}'")


test_dataset = Dataset.from_dict({"text": [user_startup_idea]})
tokenized_test = test_dataset.map(
    lambda x: tokenizer(x['text'], padding="max_length", truncation=True),
    batched=True
)

print("\n--- 2. RUNNING CLASSIFIER ---")
test_results = trainer.predict(tokenized_test)
predicted_class_id = np.argmax(test_results.predictions, axis=-1)[0]
classified_risk = reverse_label_map[predicted_class_id]

print(f"Predicted Risk Label: {classified_risk}")


if classified_risk == "Unacceptable_Risk":
    final_answer = (
        "🚨 **COMPLIANCE ALERT: UNACCEPTABLE RISK** 🚨\n\n"
        "Under the EU AI Act, systems categorized as 'Unacceptable Risk' "
        "are strictly prohibited. You cannot legally build, deploy, or place this system "
        "on the market in the EU. Therefore, no documentation or registration rules apply—the "
        "system must be abandoned or fundamentally redesigned."
    )
    print("\n[AI ACT ADVISOR RESPONSE]")
    print(final_answer)


else:
    print("\n--- 3. RETRIEVING CHUNKS ---")
    top_chunks = retrieve_and_filter_chunks(
        query=user_question,
        hybrid_retriever=hybrid_retriever,
        target_role=user_provided_role,
        target_risk=classified_risk,
        final_k=5
    )
    print(f"Retrieved {len(top_chunks)} chunks.")

    print("\n--- 4. GENERATING FINAL SUPER PROMPT ---")
    formatted_context = "\n".join([f"--- CHUNK {i+1} ---\n{c.page_content}" for i, c in enumerate(top_chunks)])
    prompt_template = build_super_prompt(user_role=user_provided_role, user_risk=classified_risk)
    final_prompt = prompt_template.format(context=formatted_context, question=user_question)

    print("\n--- 5. GENERATING FINAL LLM ANSWER ---")
    final_answer = generate_final_answer(final_prompt)

    print("\n[AI ACT ADVISOR RESPONSE]")
    print(final_answer)

Loading ULMFiT model...

--- 1. PROCESSING USER INPUT ---
Startup Idea: 'We are deploying an emotion recognition AI system in corporate workplaces and universities to constantly monitor the facial expressions of employees and students to infer their current emotional state and engagement levels.'


Map:   0%|          | 0/1 [00:00<?, ? examples/s]


--- 2. RUNNING CLASSIFIER ---


Predicted Risk Label: Unacceptable_Risk

[AI ACT ADVISOR RESPONSE]
🚨 **COMPLIANCE ALERT: UNACCEPTABLE RISK** 🚨

Under the EU AI Act, systems categorized as 'Unacceptable Risk' are strictly prohibited. You cannot legally build, deploy, or place this system on the market in the EU. Therefore, no documentation or registration rules apply—the system must be abandoned or fundamentally redesigned.


In [87]:
user_startup_idea = "We are building an AI chatbot for our e-commerce website to answer basic customer FAQs."
user_question = "Do I need to establish a quality management system and draw up technical documentation?"
user_provided_role = "provider"

print("\n--- 1. PROCESSING USER INPUT ---")
print(f"Startup Idea: '{user_startup_idea}'")


test_dataset = Dataset.from_dict({"text": [user_startup_idea]})
tokenized_test = test_dataset.map(
    lambda x: tokenizer(x['text'], padding="max_length", truncation=True),
    batched=True
)

print("\n--- 2. RUNNING CLASSIFIER ---")
test_results = trainer.predict(tokenized_test)
predicted_class_id = np.argmax(test_results.predictions, axis=-1)[0]
classified_risk = reverse_label_map[predicted_class_id]

print(f"Predicted Risk Label: {classified_risk}")


if classified_risk == "Unacceptable_Risk":
    print("\n--- 3. GUARDRAIL ACTIVATED ---")
    final_answer = (
        "🚨 **COMPLIANCE ALERT: UNACCEPTABLE RISK** 🚨\n\n"
        "Under the EU AI Act, systems categorized as 'Unacceptable Risk' "
        "are strictly prohibited. You cannot legally build, deploy, or place this system "
        "on the market in the EU. Therefore, no documentation or registration rules apply—the "
        "system must be abandoned or fundamentally redesigned."
    )
    print("\n[AI ACT ADVISOR RESPONSE]")
    print(final_answer)


else:
    print("\n--- 3. RETRIEVING CHUNKS ---")
    top_chunks = retrieve_and_filter_chunks(
        query=user_question,
        hybrid_retriever=hybrid_retriever,
        target_role=user_provided_role,
        target_risk=classified_risk,
        final_k=5
    )
    print(f"Retrieved {len(top_chunks)} chunks.")

    print("\n--- 4. GENERATING FINAL SUPER PROMPT ---")
    formatted_context = "\n".join([f"--- CHUNK {i+1} ---\n{c.page_content}" for i, c in enumerate(top_chunks)])
    prompt_template = build_super_prompt(user_role=user_provided_role, user_risk=classified_risk)
    final_prompt = prompt_template.format(context=formatted_context, question=user_question)

    print("\n--- 5. GENERATING FINAL LLM ANSWER ---")
    final_answer = generate_final_answer(final_prompt)

    print("\n[AI ACT ADVISOR RESPONSE]")
    print(final_answer)


--- 1. PROCESSING USER INPUT ---
Startup Idea: 'We are building an AI chatbot for our e-commerce website to answer basic customer FAQs.'


Map:   0%|          | 0/1 [00:00<?, ? examples/s]


--- 2. RUNNING CLASSIFIER ---


Predicted Risk Label: Limited_Risk

--- 3. RETRIEVING CHUNKS ---
Retrieved 5 chunks.

--- 4. GENERATING FINAL SUPER PROMPT ---

--- 5. GENERATING FINAL LLM ANSWER ---
Generating answer... (this might take a few seconds)

[AI ACT ADVISOR RESPONSE]
You are not subject to the obligations regarding quality management systems and technical documentation as outlined in the AI Act. 
[Article 18] states that providers of high-risk AI systems must keep certain documentation for 10 years after the AI system is placed on the market or put into service. However, this does not apply to you as a LIMITED RISK PROVIDER.

---
* **Article 5:**  [Article 5] of the AI Act defines "high-risk AI systems" and outlines the obligations for providers of such systems.
* **Recital 10:** [Recital 10] of the AI Act explains the rationale for establishing the AI Act and the need to ensure the safety and security of high-risk AI systems.


In [83]:
user_startup_idea = "We are developing an AI-driven triage system for hospital emergency rooms to automatically prioritize patients based on the severity of their symptoms and vital signs."
user_question = "What are the specific requirements for 'automatic logging' in my system, and for how long must I keep these logs available?"
user_provided_role = None

print("\n--- 1. PROCESSING USER INPUT ---")
print(f"Startup Idea: '{user_startup_idea}'")


test_dataset = Dataset.from_dict({"text": [user_startup_idea]})
tokenized_test = test_dataset.map(
    lambda x: tokenizer(x['text'], padding="max_length", truncation=True),
    batched=True
)

print("\n--- 2. RUNNING CLASSIFIER ---")
test_results = trainer.predict(tokenized_test)
predicted_class_id = np.argmax(test_results.predictions, axis=-1)[0]
classified_risk = reverse_label_map[predicted_class_id]

print(f"Predicted Risk Label: {classified_risk}")


if classified_risk == "Unacceptable_Risk":
    print("\n--- 3. GUARDRAIL ACTIVATED ---")
    final_answer = (
        "🚨 **COMPLIANCE ALERT: UNACCEPTABLE RISK** 🚨\n\n"
        "Under the EU AI Act, systems categorized as 'Unacceptable Risk' "
        "are strictly prohibited. You cannot legally build, deploy, or place this system "
        "on the market in the EU. Therefore, no documentation or registration rules apply—the "
        "system must be abandoned or fundamentally redesigned."
    )
    print("\n[AI ACT ADVISOR RESPONSE]")
    print(final_answer)


else:
    print("\n--- 3. RETRIEVING CHUNKS ---")
    top_chunks = retrieve_and_filter_chunks(
        query=user_question,
        hybrid_retriever=hybrid_retriever,
        target_role=user_provided_role,
        target_risk=classified_risk,
        final_k=5
    )
    print(f"Retrieved {len(top_chunks)} chunks.")

    print("\n--- 4. GENERATING FINAL SUPER PROMPT ---")
    formatted_context = "\n".join([f"--- CHUNK {i+1} ---\n{c.page_content}" for i, c in enumerate(top_chunks)])
    prompt_template = build_super_prompt(user_role=user_provided_role, user_risk=classified_risk)
    final_prompt = prompt_template.format(context=formatted_context, question=user_question)

    print("\n--- 5. GENERATING FINAL LLM ANSWER ---")
    final_answer = generate_final_answer(final_prompt)

    print("\n[AI ACT ADVISOR RESPONSE]")
    print(final_answer)


--- 1. PROCESSING USER INPUT ---
Startup Idea: 'We are developing an AI-driven triage system for hospital emergency rooms to automatically prioritize patients based on the severity of their symptoms and vital signs.'


Map:   0%|          | 0/1 [00:00<?, ? examples/s]


--- 2. RUNNING CLASSIFIER ---


Predicted Risk Label: High_Risk

--- 3. RETRIEVING CHUNKS ---
Retrieved 5 chunks.

--- 4. GENERATING FINAL SUPER PROMPT ---

--- 5. GENERATING FINAL LLM ANSWER ---
Generating answer... (this might take a few seconds)

[AI ACT ADVISOR RESPONSE]
The AI Act requires providers of high-risk AI systems to keep logs automatically generated by their systems. These logs are subject to specific requirements regarding their content, duration of retention, and accessibility. 
[Article 12(1)] mandates the retention of these logs for a minimum of six months, unless otherwise stipulated by national or Union law. 
[Article 22] further empowers authorized representatives to access these logs upon request from competent authorities. 
---
 
### SOURCES:
---
* [Article 12(1)]
* [Article 22]
* [Recital 69]
* [Recital 71]


In [88]:
user_startup_idea = "We are developing an AI-driven triage system for hospital emergency rooms to automatically prioritize patients based on the severity of their symptoms and vital signs."
user_question = "What are the specific requirements for 'automatic logging' in my system, and for how long must I keep these logs available?"
user_provided_role = None


print("\n--- GENERATING LLM ANSWER WITHOUT RAG AND METADATA ---")

# Construct a simple prompt using only the user's inputs and the classified risk
baseline_prompt = f"""
You are an AI Act compliance assistant.


User Question: {user_question}

"""

# Pass the un-augmented prompt directly to your generation function
baseline_answer = generate_final_answer(baseline_prompt)

print("\n[PURE LLM RESPONSE (NO RAG)]")
print(baseline_answer)


--- GENERATING LLM ANSWER WITHOUT RAG AND METADATA ---
Generating answer... (this might take a few seconds)

[PURE LLM RESPONSE (NO RAG)]
**AI Act Compliance Assistant:**

It's great you're thinking about compliance with the AI Act!  

Here's a breakdown of the requirements for 'automatic logging' and how long you need to keep those logs:

**Automatic Logging Requirements:**

* **Purpose:**  You must automatically log all data processing activities that are likely to have a significant impact on individuals' rights and freedoms. This includes:
    * **Data collection:**  What data is being collected, how is it collected, and from whom?
    * **Data processing:**  What is being done with the data (e.g., analysis, classification, storage)?
    * **Decision-making:**  How are decisions made based on the data?
    * **System interactions:**  How does the system interact with users and other systems?
* **Content:**  The logs must include:
    * **Timestamp:**  When the event occurred.
    